# GLiNER multi-v2.1 — DIMER zero-shot named-entity recognition tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gliner-ner-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gliner-ner-pipeline/blob/main/tutorials/gliner_ner_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-urchade%2Fgliner__multi--v2.1-ffcc4d?style=flat)](https://huggingface.co/urchade/gliner_multi-v2.1) [![Upstream](https://img.shields.io/badge/Upstream-urchade%2FGLiNER-181717?style=flat&logo=github&logoColor=white)](https://github.com/urchade/GLiNER) [![arXiv](https://img.shields.io/badge/arXiv-2311.08526-b31b1b.svg)](https://arxiv.org/abs/2311.08526)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot named-entity recognition with a caller-supplied label set using the pinned GLiNER multi-v2.1 weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/gliner_ner_pipeline/pipeline.py` at revision `591830bdfdcc`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `443d26d654e0324125a96bebd8e796c14ff2efe6` (~1160 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the text and the caller's entity-type names are encoded together by the mDeBERTa-v3 bidirectional encoder inside GLiNER, every candidate span is scored against every label, and spans whose **score** (a per-span confidence in [0, 1], **not a calibrated probability**) reaches the `threshold` are returned with character offsets. The label set is free text chosen by the caller at call time — that is what "zero-shot" means here — and the threshold (default `DEFAULT_THRESHOLD` = 0.5, the upstream default) is exposed and **owned by the caller**. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. **Two pinned snapshots** are involved: the GLiNER weights and the mDeBERTa tokenizer/config the encoder needs; the carried module verifies both against their own manifests before loading, and Section 3 carries both inline. What upstream supplies is the model, the encoder assets and the span-decoding library; what the carried pipeline module adds is dual-manifest verification, input validation and ceilings, offset-checked output, and the `entity_f1`, `validate_inputs` and `evaluation_report` helpers.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author a synthetic input text and label set (or upload your own), stage and digest-verify **both** immutable snapshots, validate the request into an input manifest through the pipeline's own validation stage, detect entities through the public API, read spans and scores correctly, produce an evaluation report that is `sample-sanity` with `entity_f1` only when gold spans exist and `not-measurable` otherwise, and export the entities with identifiers plus provenance.

**This notebook does not demonstrate:** entity linking or normalisation, relation extraction, coreference, document-level processing beyond `MAX_TEXT_CHARS` (the caller chunks), fine-tuning, or any calibrated confidence. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. The pinned `torch==2.14.0` install and the 1.16 GB GLiNER checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python; what character offsets are; what precision/recall/F1 over spans mean.
- **Expected warning:** while building the fast DeBERTa tokenizer from `spm.model`, `transformers==4.57.6` logs an "incorrect regex pattern … `fix_mistral_regex`" warning. It refers to a Mistral tokenizer issue, does not apply to this SentencePiece model, and is documented in the README and model card; the pipeline captures it in `load_warnings`, which Section 5 prints so you can see it is the expected one. Building that tokenizer needs `protobuf` and `sentencepiece`, both in the pinned install.
- **Data:** the default sample is a synthetic sentence and label set authored in code (the names are invented); BYOD is one UTF-8 text file plus a label list, gated off by default, at most `MAX_TEXT_CHARS` characters (the library also cuts text beyond 384 words — chunk longer documents yourself). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `urchade/gliner_multi-v2.1` snapshot (~1160 MB in total) at revision `443d26d654e0…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `gliner` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'gliner==0.2.29',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'protobuf==6.31.1',
    'sentencepiece==0.2.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'gliner-ner-pipeline',
    'repository_revision': '591830bdfdcc1fc268246f4330b3d2a77beb1975',
    'embedded_module': 'src/gliner_ner_pipeline/pipeline.py',
    'embedded_modules': ['src/gliner_ner_pipeline/pipeline.py'],
    'module_sha256': '149dccb25609b18f3670e3b739e736fa23f34ee3ba088c9ea226389cb7e1d0b3',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, gliner
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'gliner': gliner.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/gliner_ner_pipeline/` @ `591830bdfdcc`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/gliner_ner_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import hashlib
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "urchade/gliner_multi-v2.1"
MODEL_REVISION = "443d26d654e0324125a96bebd8e796c14ff2efe6"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "gliner-multi-v2.1"
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"

# The GLiNER snapshot holds only gliner_config.json + model.safetensors; its `model_name` names the
# encoder whose tokenizer and config the gliner library resolves at load time. Those files are pinned
# here as a SECOND snapshot with its own manifest (no encoder weights: model.safetensors carries them),
# and the loader points the library at that verified directory instead of the Hub or an HF cache.
ENCODER_MODEL_ID = "microsoft/mdeberta-v3-base"
ENCODER_REVISION = "a0484667b22365f84929a935b5e50a51f71f159d"
ENCODER_LICENSE = "mit"
ENCODER_KEY = "mdeberta-v3-base-tokenizer"
ENCODER_WEIGHTS_DIR = _WEIGHTS_ROOT / ENCODER_KEY
DEFAULT_THRESHOLD = 0.5  # upstream predict_entities default; the caller owns tuning it
MAX_LABELS = 25  # gliner_config.json max_types: the most entity types seen per example in training
MAX_TEXT_CHARS = 5_000  # per call; gliner_config.json max_len is 384 words, longer text is cut by the library
MAX_LABEL_CHARS = 100


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the GLiNER snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def verify_encoder_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the mDeBERTa tokenizer/config snapshot against its own manifest (second supply-chain check)."""
    root = Path(path) if path is not None else ENCODER_WEIGHTS_DIR
    return _verify_manifest(root, ENCODER_MODEL_ID, ENCODER_REVISION)


def _hub_download(
    relative_path: str, root: Path, model_id: str = MODEL_ID, revision: str = MODEL_REVISION
) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(model_id, relative_path, revision=revision, local_dir=str(root))


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
) -> list[str]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}"
        )
    fetch = downloader or (lambda rel, dst: _hub_download(rel, dst, model_id, revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch GLiNER manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(root, MODEL_ID, MODEL_REVISION, allow_download, downloader)


def stage_missing_encoder_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Same as `stage_missing_files` for the mDeBERTa tokenizer/config snapshot at ENCODER_REVISION."""
    root = Path(path) if path is not None else ENCODER_WEIGHTS_DIR
    return _stage_missing(root, ENCODER_MODEL_ID, ENCODER_REVISION, allow_download, downloader)


def entity_f1(predicted: Sequence[dict[str, Any]], gold: Sequence[dict[str, Any]]) -> dict[str, float]:
    """Exact-span micro precision/recall/F1: a hit is an identical (start, end, label) triple."""
    pred_set = {(int(e["start"]), int(e["end"]), str(e["label"])) for e in predicted}
    gold_set = {(int(e["start"]), int(e["end"]), str(e["label"])) for e in gold}
    hits = len(pred_set & gold_set)
    precision = hits / len(pred_set) if pred_set else 0.0
    recall = hits / len(gold_set) if gold_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "hits": hits}


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one text string plus 1..MAX_LABELS unique, non-empty, caller-supplied entity-type labels",
    "text_chars": [1, MAX_TEXT_CHARS],
    "labels": [1, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "threshold": [0.0, 1.0],
    "preprocessing": (
        "the gliner library tokenizes with the pinned mDeBERTa-v3 tokenizer and truncates at "
        "gliner_config.json max_len = 384 words, so text beyond that is silently cut; returned spans are "
        "character offsets into the exact string you passed"
    ),
}


def _check_inputs(text: Any, labels: Any, threshold: Any) -> tuple[str, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``GLiNERPipeline.detect`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    if not isinstance(text, str):
        raise TypeError(f"text must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError("text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"text has {len(text)} chars; ceiling is {MAX_TEXT_CHARS} (chunk it first)")
    if isinstance(labels, str | bytes) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of str")
    if not 1 <= len(labels) <= MAX_LABELS:
        raise ValueError(f"labels must hold 1..{MAX_LABELS} items, got {len(labels)}")
    for i, label in enumerate(labels):
        if not isinstance(label, str) or not label.strip():
            raise TypeError(f"labels[{i}] must be a non-empty str")
        if len(label) > MAX_LABEL_CHARS:
            raise ValueError(f"labels[{i}] has {len(label)} chars; ceiling is {MAX_LABEL_CHARS}")
    if len(set(labels)) != len(labels):
        raise ValueError("labels must be unique")
    bad_type = isinstance(threshold, bool) or not isinstance(threshold, int | float)
    if bad_type or not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be a number in [0, 1]")
    return text, list(labels), float(threshold)


def validate_inputs(
    text: str,
    labels: Sequence[str],
    threshold: float = DEFAULT_THRESHOLD,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``. Both pinned snapshot
    identities are recorded, because this pipeline verifies two (GLiNER and its mDeBERTa encoder).
    """
    checked_text, checked_labels, checked_threshold = _check_inputs(text, labels, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one text)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "text-0",
                "chars": len(checked_text),
                "words": len(checked_text.split()),
            }
        ],
        "labels": checked_labels,
        "threshold": checked_threshold,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "encoder_model_id": ENCODER_MODEL_ID,
        "encoder_revision": ENCODER_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    gold: Sequence[Mapping[str, Any]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``gold`` spans — dicts carrying ``start``, ``end`` and ``label`` — the report carries the
    repository's ``entity_f1`` (exact-span micro precision/recall/F1) as sample-sanity evidence;
    without them the verdict is ``not-measurable`` and the report says what labelled data would make
    the task measurable.
    """
    entities = list(result["entities"])
    base = {
        "task": "zero-shot named-entity recognition with a caller-supplied label set",
        "decision_rule": (
            f"a span is kept when its score reaches the caller's threshold "
            f"(default DEFAULT_THRESHOLD={DEFAULT_THRESHOLD}); the pipeline ships no tuned operating point"
        ),
        "score_semantics": (
            "each entity score is the model's own uncalibrated span score, not a probability that the "
            "span is correct"
        ),
        "threshold": result.get("threshold", DEFAULT_THRESHOLD),
        "labels": list(result.get("labels", [])),
        "sample_kind": sample_kind,
        "n_entities": len(entities),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "encoder_model_id": ENCODER_MODEL_ID,
        "encoder_revision": ENCODER_REVISION,
    }
    if gold is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no gold spans were supplied for the evaluated text",
            "needs": (
                "gold (start, end, label) spans over the same label set on text from your domain, scored "
                "with entity_f1 across enough documents to state a dispersion; exact-span matching also "
                "requires the annotation guideline to agree with the model's span boundaries"
            ),
        }
    gold_spans = list(gold)
    scores = entity_f1(entities, gold_spans)
    return {
        **base,
        "metrics": [
            {
                "id": "entity_f1",
                "value": scores["f1"],
                "precision": scores["precision"],
                "recall": scores["recall"],
                "hits": scores["hits"],
                "n_predicted": len(entities),
                "n_gold": len(gold_spans),
                "matching": "exact (start, end, label) triple",
                "estimation": "one text, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"{len(gold_spans)} gold span(s) on one tutorial text; exact-span sanity evidence, not an "
            "NER benchmark"
        ),
        "needs": (
            "a labelled span set from the deployment domain, with the same label vocabulary and the same "
            "boundary convention, for any generalisable precision/recall/F1 claim"
        ),
    }


def _local_encoder_class(root: Path, encoder_dir: Path) -> type:
    """The concrete GLiNER class for this config, with `model_name` redirected to the verified encoder
    directory so the library reads the tokenizer and AutoConfig from disk (local_files_only, no cache)."""
    from gliner import GLiNER

    config_dict = json.loads((root / "gliner_config.json").read_text(encoding="utf-8"))
    base = GLiNER._get_gliner_class(GLiNER._config_from_dict(config_dict))

    class LocalEncoderGLiNER(base):
        @classmethod
        def _load_config(cls, config_file: Path, **overrides: Any) -> Any:
            config = super()._load_config(config_file, **overrides)
            if config.model_name != ENCODER_MODEL_ID:
                raise ValueError(f"gliner_config.json names encoder {config.model_name!r}, not the pinned id")
            config.model_name = str(encoder_dir)
            return config

    return LocalEncoderGLiNER


@dataclass
class GLiNERPipeline:
    """Zero-shot NER. `_runner(text, labels, threshold)` returns gliner entity dicts."""

    _runner: Callable[[str, list[str], float], list[dict[str, Any]]]
    device: str
    load_warnings: list[str] = field(default_factory=list)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        encoder_dir: str | Path | None = None,
    ) -> GLiNERPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        enc = Path(encoder_dir) if encoder_dir is not None else ENCODER_WEIGHTS_DIR
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            if (root / MANIFEST_NAME).is_file():
                stage_missing_files(root, allow_download=allow_download)
                verify_snapshot(root)
                stage_missing_encoder_files(enc, allow_download=allow_download)
                verify_encoder_snapshot(enc)
                loader = _local_encoder_class(root, enc)
                model = loader.from_pretrained(
                    str(root), model_dir=str(root), local_files_only=True, map_location="cpu"
                )
            elif allow_download:
                from gliner import GLiNER  # Hub path: the library fetches the encoder assets unpinned

                model = GLiNER.from_pretrained(MODEL_ID, revision=MODEL_REVISION, map_location="cpu")
            else:
                raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Validate and verify the snapshots before importing model libraries.
        import torch
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]

        def runner(text: str, labels: list[str], threshold: float) -> list[dict[str, Any]]:
            with torch.inference_mode():
                return model.predict_entities(text, labels, threshold=threshold)

        return cls(runner, resolved_device, messages)

    def detect(
        self,
        text: str,
        labels: Sequence[str],
        threshold: float = DEFAULT_THRESHOLD,
    ) -> dict[str, Any]:
        """Extract spans for the caller-supplied `labels`; `threshold` is the upstream score cutoff."""
        text, labels, threshold = _check_inputs(text, labels, threshold)
        raw = self._runner(text, labels, threshold)
        entities = []
        for e in raw:
            start, end = int(e["start"]), int(e["end"])
            if not 0 <= start < end <= len(text) or e["label"] not in labels:
                raise RuntimeError(f"backend returned an invalid entity: {e}")
            entities.append(
                {
                    "text": text[start:end],
                    "label": str(e["label"]),
                    "start": start,
                    "end": end,
                    "score": float(e["score"]),
                }
            )
        return {
            "entities": entities,
            "n_entities": len(entities),
            "labels": list(labels),
            "threshold": threshold,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "encoder_model_id": ENCODER_MODEL_ID,
            "encoder_revision": ENCODER_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `443d26d654e0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `mdeberta-v3-base-tokenizer` (4 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "gliner-multi-v2.1",
  "modelId": "urchade/gliner_multi-v2.1",
  "revision": "443d26d654e0324125a96bebd8e796c14ff2efe6",
  "files": [
    {
      "path": "README.md",
      "bytes": 4770,
      "sha256": "820125f2ea897716cc38645d94f73f390ec9dfd392980c436be1d25c2e106b55"
    },
    {
      "path": "gliner_config.json",
      "bytes": 477,
      "sha256": "e25f61d91620df84aae8076811ee592e926e94d341b82e7bc1be359718f83017"
    },
    {
      "path": "model.safetensors",
      "bytes": 1155830112,
      "sha256": "2100142f31627531497850659dcb3821c99d5e71c08a8e01a98e4b11ef32a199"
    }
  ],
  "totalBytes": 1155835359
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

ENCODER_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mdeberta-v3-base-tokenizer",
  "modelId": "microsoft/mdeberta-v3-base",
  "revision": "a0484667b22365f84929a935b5e50a51f71f159d",
  "role": "encoder-tokenizer-and-config for gliner-multi-v2.1 (no weights)",
  "files": [
    {
      "path": "README.md",
      "bytes": 3667,
      "sha256": "90d5e535e04fc9ac4f39336ed33c1adc7d0136b8ce3bbe211c18a0d47b72342f"
    },
    {
      "path": "config.json",
      "bytes": 579,
      "sha256": "bcffcd343dc5efa5ef2d5a58d2b405eed108f01cc45b48d0a907b333ec41801f"
    },
    {
      "path": "spm.model",
      "bytes": 4305025,
      "sha256": "13c8d666d62a7bc4ac8f040aab68e942c861f93303156cc28f5c7e885d86d6e3"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 52,
      "sha256": "3f3978e0c036f2c2588cac34a6047cbb0af0b0dc1814254e291028529805496d"
    }
  ],
  "totalBytes": 4309323
}

if (ENCODER_MANIFEST['modelId'], ENCODER_MANIFEST['revision']) != (ENCODER_MODEL_ID, ENCODER_REVISION):
    raise RuntimeError('inline mdeberta-v3-base-tokenizer manifest does not name the identity carried by the pipeline module')
ENCODER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(ENCODER_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(ENCODER_MANIFEST, handle, indent=2)
fetched_mdeberta_v3_base_tokenizer = stage_missing_encoder_files(ENCODER_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(ENCODER_WEIGHTS_DIR), 'fetched': fetched_mdeberta_v3_base_tokenizer})
_extra = verify_encoder_snapshot(ENCODER_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_mdeberta_v3_base_tokenizer': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: one English sentence and a four-type label set written in this cell, so it needs no download and names no real person (the names are invented). It ships **no gold annotations** — the sentence was written to contain obvious entities, but the notebook does not assert where they are — so the spans it produces are smoke/sanity evidence that the code path works, never an accuracy measurement and never benchmark evidence. If you want a metric, paste gold annotations into `GOLD_JSON` (a JSON list of `{"start", "end", "label"}` objects with character offsets into the text, matching the label names exactly); Section 7 then scores exact-span micro precision/recall/F1 with the repository's `entity_f1`. Leave it empty and the evaluation report is `not-measurable`.

BYOD is optional and disabled by default. Nothing is validated in this cell beyond the shape of `GOLD_JSON` (which is notebook-side reference data, not pipeline input) — the next section hands the text, the labels and the threshold to the pipeline's own validation stage, which is the only checker for them.

In [ ]:
import hashlib
import json

USE_BYOD = False  # @param {type:"boolean"}
LABELS = 'person, organization, location, date'  # @param {type:"string"}
THRESHOLD = 0.5  # @param {type:"number"}
GOLD_JSON = ''  # @param {type:"string"}

labels = [label.strip() for label in LABELS.split(',') if label.strip()]
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    text = uploaded[sample_name].decode('utf-8').strip()
    sample_kind = 'BYOD upload'
else:
    text = 'Elena Marquez joined the Lakeside Research Institute in Wellington on 3 March 2021 after leaving Orion Analytics.'
    sample_name = 'synthetic_sentence'
    sample_kind = 'synthetic (authored in this cell; invented names)'
gold = json.loads(GOLD_JSON) if GOLD_JSON.strip() else None
if gold is not None:
    for index, item in enumerate(gold):
        if not (isinstance(item, dict) and {'start', 'end', 'label'} <= set(item)):
            raise ValueError(f'GOLD_JSON[{index}] must be an object with start, end and label')
        if not (0 <= int(item['start']) < int(item['end']) <= len(text)) or item['label'] not in labels:
            raise ValueError(f'GOLD_JSON[{index}] has offsets outside the text or a label not in LABELS: {item}')
sample_sha256 = hashlib.sha256(text.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(text), 'words': len(text.split()), 'labels': labels, 'threshold': THRESHOLD, 'has_gold': gold is not None, 'text_sha256': sample_sha256})
print(text)

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `detect` applies — a non-empty `str` of at most `MAX_TEXT_CHARS` characters, 1..`MAX_LABELS` unique non-empty labels of at most `MAX_LABEL_CHARS` characters each, and a `threshold` in [0, 1] — and returns an **input manifest** naming the schema and ceilings, the text's observed character and word counts, the label set, the threshold, the verdict, and **both** pinned identities (this pipeline verifies two snapshots). The manifest is written to `outputs/gliner_ner_input_manifest.json`. To show what rejection looks like, the cell also validates a duplicated label set and records the pipeline's own error message as a finding.

**What can change your text:** the GLiNER library cuts input beyond 384 words (`max_len` in the snapshot config) without reporting where — the word count is printed so you can see whether the sample is near that limit; the notebook itself does not alter the text.

In [ ]:
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'library_word_limit': 384, 'DEFAULT_THRESHOLD': DEFAULT_THRESHOLD}})
input_manifest = validate_inputs(text, labels, THRESHOLD, names=[sample_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(text, labels + labels[:1], THRESHOLD)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-label-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/gliner_ner_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Detect entities and read the scores correctly

`detect(text, labels, threshold=...)` returns `entities` — a list of `{text, label, start, end, score}` with character offsets into the input (the pipeline checks every span lies inside the text and carries one of your labels) — plus `n_entities`, the `labels` and `threshold` used, and both model identities. **Score semantics:** each `score` is the library's per-span confidence in [0, 1] for that span-label pair; it is not a calibrated probability, and the only decision the pipeline makes is the cutoff `score >= threshold`. The default 0.5 is the upstream default, not a validated operating point: lowering it returns more (and less certain) spans, raising it fewer; the caller owns choosing it on their own annotated data. The captured `load_warnings` are printed here — expect exactly the `fix_mistral_regex` notice described in the prerequisites. The printed seconds are measured on this runtime for this one text and include the first-call warm-up.

In [ ]:
import time

print({'load_warnings': pipe.load_warnings})
started = time.perf_counter()
result = pipe.detect(text, labels, threshold=THRESHOLD)
elapsed = time.perf_counter() - started
entities = result['entities']
checks = {
    'offsets_inside_text': all(0 <= e['start'] < e['end'] <= len(text) for e in entities),
    'span_text_matches_offsets': all(text[e['start']:e['end']] == e['text'] for e in entities),
    'labels_from_request': all(e['label'] in labels for e in entities),
    'scores_at_or_above_threshold': all(THRESHOLD <= e['score'] <= 1.0 for e in entities),
}
if not all(checks.values()):
    raise RuntimeError(f'detect output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'entities'})
print({'seconds': round(elapsed, 3), 'checks': checks})
for index, entity in enumerate(entities):
    print(f"{index:>2}. [{entity['start']:>3}:{entity['end']:<3}] {entity['label']:<14} score {entity['score']:.3f}  {entity['text']}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. The repository's only metric helper is `entity_f1(predicted, gold)` — exact-span micro precision, recall and F1, where a hit is an identical `(start, end, label)` triple — the standard NER measure, which credits nothing for a span that is off by one character or carries a different label. It applies only when gold annotations exist. The synthetic sample has none, so the verdict is `not-measurable` and the report states what would make the task measurable: gold `(start, end, label)` spans over the same label set, on text from your domain, across enough documents to state a dispersion — and an annotation guideline that agrees with the model's span boundaries. When you supply `GOLD_JSON` the verdict becomes `sample-sanity` with one `entity_f1` metric: a single-text tutorial figure with no dispersion estimate, not a benchmark result. No baseline is reported: a trivial baseline (no entities) has F1 = 0 by construction and teaches nothing without gold. The report is written to `outputs/gliner_ner_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, gold, sample_kind=sample_kind)
with open('outputs/gliner_ner_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No gold annotations were supplied, so entity_f1 is not computed; the spans above are sanity evidence only.')

## 8. Export entities and provenance

Two files are written under `outputs/`: `gliner_ner_result.json` with the input text and its digest, the label set and threshold, an `entities` list with an index per span plus its text, label, character offsets and score (so every span maps back to its input), the evaluation report, the input manifest, the gold spans when supplied, the sanity checks, the ceilings in force, the notebook's source (repository, revision, embedded module digest, generator), **both** model identifiers and immutable revisions, the model licences, the captured load warnings, and the runtime identity (Python, `torch`, `gliner`, `transformers`, device, precision); and `gliner_ner_entities.csv` with explicit `index`, `start`, `end`, `label`, `score` and `text` columns so span order and offsets survive downstream use. No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

payload = {
    'text': text,
    'labels': result['labels'],
    'threshold': result['threshold'],
    'entities': [{'index': index, **entity} for index, entity in enumerate(entities)],
    'n_entities': result['n_entities'],
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'gold': gold,
    'sanity_checks': checks,
    'ceilings': {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'library_word_limit': 384, 'DEFAULT_THRESHOLD': DEFAULT_THRESHOLD},
    'sample': {'name': sample_name, 'kind': sample_kind, 'text_sha256': sample_sha256},
    'seconds': round(elapsed, 3),
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'encoder_model_id': ENCODER_MODEL_ID,
    'encoder_revision': ENCODER_REVISION,
    'encoder_license': ENCODER_LICENSE,
    'load_warnings': pipe.load_warnings,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'gliner': gliner.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/gliner_ner_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/gliner_ner_entities.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['index', 'start', 'end', 'label', 'score', 'text'])
    for index, entity in enumerate(entities):
        writer.writerow([index, entity['start'], entity['end'], entity['label'], f"{entity['score']:.6f}", entity['text']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Each returned span is a model prediction for a label name you chose: the `score` is a per-span confidence in [0, 1], not a calibrated probability, and the only decision rule is the caller-owned `threshold` (default 0.5, the upstream default, not a validated operating point). On the synthetic sentence the spans are plumbing evidence only and the evaluation report is `not-measurable`; an `entity_f1` figure on one text has no dispersion and generalises to nothing. The label names are part of the input — different wordings of the same concept give different spans — and the checkpoint's multilingual quality varies by language and domain in ways this notebook does not measure. Text beyond 384 words is cut silently by the library (the notebook prints the word count; chunk long documents), nested or overlapping entities may be lost, and entity linking, normalisation, relations and coreference are not provided. Two supply chains are involved — the GLiNER weights and the mDeBERTa encoder assets — and both were digest-verified against inline manifests before loading. Inference is deterministic given the same weights, device and library versions; CPU and CUDA scores can differ slightly and move borderline spans across the threshold.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can stage and digest-verify both pinned snapshots, validate the demonstrated request against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, extraction quality on any domain or language, a validated threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/gliner-multi-v2.1/` (or `weights/mdeberta-v3-base-tokenizer/`) and rerun Section 3. A `fix_mistral_regex` warning: expected and harmless (see prerequisites). An `ImportError` naming `protobuf` while the DeBERTa tokenizer is built: the pinned install provides it — restart the runtime and rerun from the top. A `ValueError` naming `MAX_TEXT_CHARS`, `MAX_LABELS`, `MAX_LABEL_CHARS` or the threshold in Section 5: fix the form values or chunk the text and rerun from Section 4. Zero entities returned: lower `THRESHOLD`, or check that your label names describe the entities in plain words.

**Next experiments.** Paste gold annotations for the default sentence into `GOLD_JSON` to see the verdict switch to `sample-sanity` and how exact-span F1 penalises a boundary that is off by one character; sweep `THRESHOLD` over 0.3–0.7 on a text you can judge and watch precision and recall trade off; reword a label (for example `city` versus `location`) and observe how the spans change. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/MODEL_CARD.md
- Weight provenance (both snapshots): https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/urchade/gliner_multi-v2.1
- Encoder assets: https://huggingface.co/microsoft/mdeberta-v3-base
- Upstream code: https://github.com/urchade/GLiNER
- GLiNER paper: https://arxiv.org/abs/2311.08526